# Deep Batch Active Learning by Diverse, Uncertain Gradient Lower Bounds

**Jordan T. Ash, Chicheng Zhang, Akshay Krishnamurthy, John Langford, Alekh Agarwal (2020)**

This notebook demonstrates the BADGE (Batch Active learning by Diverse Gradient Embeddings) method from the paper. BADGE selects diverse, uncertain batches for active learning by computing gradient embeddings with respect to the last layer (using hallucinated labels) and sampling via k-MEANS++ seeding, which automatically balances uncertainty and diversity without hyperparameters.

**What this notebook gives you:**
- A working implementation of BADGE's gradient embedding computation and k-MEANS++ batch selection
- A complete active learning pipeline: data loading, model training, acquisition loop, and learning curve visualization
- Per-component demonstrations of the algorithm's key pieces (gradient embeddings, k-MEANS++ seeding)

**Two ways to use this:**
1. **Run as-is:** Execute the cells below to see BADGE in action on a smoke-sized MNIST subset (~5 minutes end-to-end).
2. **Import from your own code:** Use the `select_batch` function from `method/method.py` with your own model and unlabeled pool:
   ```python
   from method import select_batch
   batch_indices = select_batch(model, x_unlabeled, batch_size=100, seed=42)
   ```

**What this notebook does NOT do:**
- This is a reference implementation, not a benchmark reproduction. The paper's empirical claims (pairwise penalty matrix, CDF comparisons) require running multiple methods across multiple datasets, architectures, batch sizes, and seeds. Defer to the paper for those results.
- This notebook runs a single seed on a smoke-sized dataset to fit within ~5 minutes. For publication-quality results, use the full datasets (MNIST 60K, CIFAR-10 60K, SVHN 73K) with 5 repeated runs per configuration.

## Contents

- [0. Install dependencies (first run only)](#sec-0-install-dependencies-first-run-only)
- [1. Setup](#sec-1-setup)
- [2. Parameters](#sec-2-parameters)
  - [Optional: scale up to paper-faithful values](#sec-optional-scale-up-to-paper-faithful-values)
- [3. The setup pieces](#sec-3-the-setup-pieces)
  - [3.1 Data](#sec-31-data)
  - [3.2 Model](#sec-32-model)
  - [3.3 Training](#sec-33-training)
  - [3.4 Bootstrap labeled set](#sec-34-bootstrap-labeled-set)
- [4. The BADGE method ⭐](#sec-4-the-badge-method)
  - [4.1 Intuition](#sec-41-intuition)
  - [4.2 Gradient Embeddings ⭐](#sec-42-gradient-embeddings)
  - [4.3 k-MEANS++ Seeding](#sec-43-k-means-seeding)
  - [4.4 Putting it together](#sec-44-putting-it-together)
- [5. Running active learning end-to-end](#sec-5-running-active-learning-end-to-end)
  - [5.1 The acquisition loop](#sec-51-the-acquisition-loop)
  - [5.2 Learning curve](#sec-52-learning-curve)
- [6. Use your own data](#sec-6-use-your-own-data)

<a id="sec-0-install-dependencies-first-run-only"></a>

## 0. Install dependencies (first run only)

Run this cell once to install the required packages. Subsequent runs are no-ops if packages are already installed.

In [ ]:
%pip install -r requirements.txt

<a id="sec-1-setup"></a>

## 1. Setup

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import torch

from method import (
    build_model,
    compute_gradient_embeddings,
    kmeans_plus_plus_seeding,
    load_data,
    select_batch,
    train_from_scratch,
)

# Master seed for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

<a id="sec-2-parameters"></a>

## 2. Parameters

Parameters are sourced from three places:
- **paper**: Directly from the paper's experimental setup (Section 4, Experiments).
- **system_default**: Reasonable defaults chosen by the pipeline when the paper value is impractical for demo scale (e.g., pool_size reduced from 60K to 800 to fit runtime budget).
- **system_inferred**: Values inferred from the paper's description when not explicitly stated (e.g., hidden_dim=256 for image data MLP).

The table below shows each parameter's provenance, value, and whether it's used in this notebook's code cells.

| Parameter | Variable from paper | Value from paper | Paper value | System value | Used? | Notes |
|---|:-:|:-:|---|---|:-:|---|
| `batch_size` | ✅ | ✅ | 100 | — | ✅ | Paper uses batch_size=100 per data_setup. |
| `num_rounds` | ✅ | ❌ | 100 | 5 | ✅ | Paper runs 100 rounds (~10,000 labels at batch_size=100). At smoke scale we... |
| `initial_labeled` | ✅ | ✅ | 100 | — | ✅ | Paper bootstraps with 100 uniformly-random labeled examples. |
| `pool_size` | ✅ | ❌ | 'full training set' | 800 | ✅ | Paper uses the full training set (typically 60k–73k samples) as the... |
| `learning_rate` | ✅ | ✅ | 0.001 | — | ✅ | Adam optimizer at 0.001. Paper: '0.001 for image data, 0.0001 for non-image... |
| `max_epochs` | ❌ | ❌ | — | 8 | ✅ | Paper trains until training accuracy ≥ 99% with no explicit epoch cap. At... |
| `train_until_accuracy` | ✅ | ✅ | 0.99 | — | ✅ | Paper: train until training accuracy exceeds 99%. |
| `hidden_dim` | ❌ | ❌ | — | 256 | ✅ | Paper uses 1024 for tabular MLP and 256 for image data with ResNet-18 /... |

In [ ]:
params = {
    "batch_size": {
        "value": 100,
        "source": 'paper',
        "paper_section": 'Section 4, Experiments',
        "note": 'Paper uses batch_size=100 per data_setup.',
    },
    "num_rounds": {
        "value": 5,
        "source": 'system_default',
        "paper_value": 100,
        "reasoning": (
            "Paper runs 100 rounds (~10,000 labels at batch_size=100). At smoke "
            "scale we run 5 rounds (~500 labels) — enough to see a learning curve "
            "emerge while keeping the acquisition loop (which re-trains and "
            "re-scores the pool every round) well within the smoke gate's per-cell "
            "time budget on a laptop CPU."
        ),
    },
    "initial_labeled": {
        "value": 100,
        "source": 'paper',
        "paper_section": 'Section 4, Experiments',
        "note": 'Paper bootstraps with 100 uniformly-random labeled examples.',
    },
    "pool_size": {
        "value": 800,
        "source": 'system_default',
        "paper_value": 'full training set',
        "reasoning": (
            "Paper uses the full training set (typically 60k–73k samples) as the "
            "unlabeled pool. We subsample to 800 so each round's gradient-embedding "
            "pass over the whole pool finishes within the smoke gate's per-cell "
            "timeout (the pool is re-scored every round, so this is the dominant "
            "cost of the acquisition loop). Pool ratio (total_budget / pool_size) ≈ "
            "0.75. This exceeds the paradigm field guide's "
            "`demo_scale.max_budget_to_pool_ratio` of 0.05 — methods may converge "
            "before diversity is meaningfully tested. Increase pool_size (or reduce "
            "num_rounds × batch_size) for cleaner diversity-baseline comparisons."
        ),
    },
    "learning_rate": {
        "value": 0.001,
        "source": 'paper',
        "paper_section": 'Section 4, Experiments',
        "note": (
            "Adam optimizer at 0.001. Paper: '0.001 for image data, 0.0001 for "
            "non-image (OpenML) data'. Selected image-data value."
        ),
    },
    "max_epochs": {
        "value": 8,
        "source": 'system_inferred',
        "reasoning": (
            "Paper trains until training accuracy ≥ 99% with no explicit epoch cap. "
            "At smoke scale the labeled sets are tiny (≤ ~500 examples) and "
            "`train_until_accuracy` usually stops well before the cap; 8 is a low "
            "safety bound that keeps per-round training fast enough for the "
            "acquisition loop to finish within the smoke gate's per-cell timeout. "
            "Increase it for full-scale training."
        ),
    },
    "train_until_accuracy": {
        "value": 0.99,
        "source": 'paper',
        "paper_section": 'Section 4, Experiments',
        "note": 'Paper: train until training accuracy exceeds 99%.',
    },
    "hidden_dim": {
        "value": 256,
        "source": 'system_inferred',
        "reasoning": (
            "Paper uses 1024 for tabular MLP and 256 for image data with ResNet-18 "
            "/ VGG-11 (per critical_requirements.model). We're running an MLP on "
            "flattened image data — neither paper config matches exactly. Picked "
            "256 as the closer match for image data; using 1024 would slow training "
            "without a clear quality gain at smoke scale."
        ),
    },
}


def unpack(p: dict) -> dict:
    """Strip provenance and return a flat name -> value dict."""
    return {k: v["value"] for k, v in p.items() if v.get("used_in_notebook", True)}


cfg = unpack(params)

<a id="sec-optional-scale-up-to-paper-faithful-values"></a>

### Optional: scale up to paper-faithful values

To run closer to the paper's full scale, uncomment and adjust the values below. Note that true paper-faithfulness also requires swapping the bundled MLP for the paper's ResNet-18 or VGG-11 architecture for image datasets.

```python
# cfg.update({
#     "num_rounds": 100,      # paper runs 100 rounds (~10,000 labels at batch_size=100)
#     "pool_size": 60000,     # full MNIST training set
# })
# ```

<a id="sec-3-the-setup-pieces"></a>

## 3. The setup pieces

BADGE works on top of standard supervised classification — model + training + labeled data. The pieces below are paper protocol but **not** the paper's contribution; skim and move on to §4 for the algorithm itself.

**Architecture note:** The bundled MLP implements `forward_with_embedding(x) -> (logits, penultimate_embedding)`, which is essential for BADGE's gradient embedding computation (**Eq. 1, Section 3**). When swapping architectures, ensure this method is present.

<a id="sec-31-data"></a>

### 3.1 Data

In [ ]:
x_pool, y_pool, x_test, y_test = load_data(pool_size=cfg["pool_size"], n_test=1000, seed=SEED)

print(f"Pool: x_pool={x_pool.shape}, y_pool={y_pool.shape}")
print(f"Test: x_test={x_test.shape}, y_test={y_test.shape}")
print(f"Number of classes: {y_pool.unique().tolist()}")

<a id="sec-32-model"></a>

### 3.2 Model

In [ ]:
input_dim = x_pool.shape[1]
n_classes = len(y_pool.unique())

model = build_model(input_dim=input_dim, n_classes=n_classes, hidden_dim=cfg["hidden_dim"])
print(model)

<a id="sec-33-training"></a>

### 3.3 Training

The paper uses Adam optimizer with learning rate 0.001 for image data, cross-entropy loss, and trains until 99% training accuracy (**Section 4, Experiments**). Models are retrained from scratch at each AL round (no warm-starting).

No code cell here — training happens per-round in §3.4 (bootstrap) and §5.1 (the acquisition loop).

<a id="sec-34-bootstrap-labeled-set"></a>

### 3.4 Bootstrap labeled set

Initialize with `cfg["initial_labeled"]` random examples from the pool, then train a warmup model. This warmup model is used in §4 to demonstrate the algorithm's components.

In [ ]:
rng = np.random.default_rng(SEED)
initial_indices = rng.choice(len(x_pool), size=cfg["initial_labeled"], replace=False)
labeled_idx = np.asarray(initial_indices)
unlabeled_idx = np.setdiff1d(np.arange(len(x_pool)), labeled_idx)

x_train = x_pool[labeled_idx]
y_train = y_pool[labeled_idx]

model = build_model(input_dim=input_dim, n_classes=n_classes, hidden_dim=cfg["hidden_dim"])
model = train_from_scratch(
    model, x_train, y_train,
    learning_rate=cfg["learning_rate"],
    max_epochs=cfg["max_epochs"],
    train_until_accuracy=cfg["train_until_accuracy"],
    seed=SEED
)

# Evaluate warmup model
with torch.no_grad():
    model.eval()
    logits = model(x_test)
    warmup_acc = (logits.argmax(dim=1) == y_test).float().mean().item()
    print(f"Warmup model test accuracy: {warmup_acc:.4f}")

<a id="sec-4-the-badge-method"></a>

## 4. The BADGE method ⭐

This is the paper's contribution (**Algorithm 1, Section 3**). BADGE computes gradient embeddings for unlabeled examples using the model's predicted (hallucinated) label, then uses k-MEANS++ seeding to select a diverse batch of high-magnitude gradients.

<a id="sec-41-intuition"></a>

### 4.1 Intuition

The core insight of BADGE is to measure **uncertainty** through gradient magnitude and **diversity** through gradient direction.

**Uncertainty via gradient magnitude:** If the model is uncertain about an example, knowing its true label would induce a large gradient update. BADGE computes this gradient using the model's own prediction (the "hallucinated label") as a proxy. Proposition 1 shows this gives a *lower bound* on the gradient norm for any true label, providing a conservative uncertainty estimate (**Proposition 1, Section 3**).

**Diversity via gradient direction:** Selecting only high-uncertainty examples leads to redundant batches (near-duplicates). BADGE avoids this by selecting examples whose gradient embeddings span diverse directions in parameter space, ensuring the batch induces varied model updates.

**Automatic trade-off via k-MEANS++:** The k-MEANS++ seeding algorithm samples points with probability proportional to squared distance from the nearest already-selected center. This naturally favors both high-magnitude points (uncertainty) and points far from already-selected ones (diversity), without requiring hyperparameters to tune the trade-off (**Algorithm 2, Appendix A**).

<a id="sec-42-gradient-embeddings"></a>

### 4.2 Gradient Embeddings ⭐

The gradient embedding for example x is computed as (**Eq. 1, Section 3**):

$$g_x^i = (p_i - \mathbb{1}\{\hat{y} = i\}) \cdot z(x; V)$$

where:
- $p = \text{softmax}(W \cdot z(x; V))$ is the predicted probability vector
- $\hat{y} = \arg\max_i p_i$ is the hallucinated label (model's prediction)
- $z(x; V)$ is the penultimate layer output
- The embedding is the concatenation over all classes $i \in [K]$

The embedding has shape $(N, K \cdot d)$ where $K$ is the number of classes and $d$ is the hidden dimension.

In [ ]:
# Compute gradient embeddings for a small subset of the unlabeled pool
subset_size = 100
subset_indices = rng.choice(len(unlabeled_idx), size=subset_size, replace=False)
x_subset = x_pool[unlabeled_idx[subset_indices]]

gradient_embeddings = compute_gradient_embeddings(model, x_subset)
print(f"Gradient embeddings shape: {gradient_embeddings.shape}")
print(f"Expected shape: ({subset_size}, {n_classes} * {cfg['hidden_dim']}) = ({subset_size}, {n_classes * cfg['hidden_dim']})")

# Show the magnitude (L2 norm) of a few gradient embeddings
# Larger magnitude = higher uncertainty
magnitudes = gradient_embeddings.norm(dim=1)
print(f"\nGradient embedding magnitudes (uncertainty proxy):")
print(f"  Min: {magnitudes.min().item():.4f}")
print(f"  Max: {magnitudes.max().item():.4f}")
print(f"  Mean: {magnitudes.mean().item():.4f}")

<a id="sec-43-k-means-seeding"></a>

### 4.3 k-MEANS++ Seeding

k-MEANS++ seeding sequentially selects $k$ centers from the ground set, where each new center is sampled with probability proportional to its squared distance from the nearest already-selected center (**Algorithm 2, Appendix A**).

Algorithm:
1. Sample the first center uniformly at random
2. For $t = 2$ to $k$:
   - Compute $D_t(x) = \min_{c \in C_{t-1}} \|x - c\|_2$ for all $x$
   - Sample $c_t$ with probability $D_t(x)^2 / \sum_x D_t(x)^2$
3. Return the set of $k$ centers

This is preferred over k-DPP (which has similar statistical performance) due to computational efficiency: k-MEANS++ is $O(k \cdot N \cdot d)$ vs. k-DPP's high-order polynomial complexity (**Figure 1, Appendix G**).

In [ ]:
# Demonstrate k-MEANS++ seeding on the gradient embeddings
batch_size = 10
rng_kmeans = np.random.default_rng(SEED)

selected_positions = kmeans_plus_plus_seeding(
    gradient_embeddings.detach().cpu().numpy(),
    batch_size=batch_size,
    rng=rng_kmeans
)

print(f"Selected positions (into subset): {selected_positions}")
print(f"Number selected: {len(selected_positions)}")

# Show that selected points have high magnitude AND are diverse
selected_embeddings = gradient_embeddings[selected_positions]
selected_magnitudes = selected_embeddings.norm(dim=1)
print(f"\nSelected embeddings' magnitudes: {selected_magnitudes.tolist()}")

# Compute pairwise distances to show diversity
from scipy.spatial.distance import cdist
pairwise_dists = cdist(selected_embeddings.cpu().numpy(), selected_embeddings.cpu().numpy(), metric='euclidean')
print(f"\nPairwise distances between selected embeddings (diversity):")
print(f"  Min (off-diagonal): {np.min(np.triu(pairwise_dists, k=1)):.4f}")
print(f"  Mean (off-diagonal): {np.mean(np.triu(pairwise_dists, k=1)):.4f}")

<a id="sec-44-putting-it-together"></a>

### 4.4 Putting it together

The `select_batch` function composes the two components above:

```python
def select_batch(model, x_unlabeled, batch_size, seed):
    # 1. Compute gradient embeddings for all unlabeled examples
    gradient_embeddings = compute_gradient_embeddings(model, x_unlabeled)

    # 2. Select diverse batch using k-MEANS++ seeding
    selected_indices = kmeans_plus_plus_seeding(
        gradient_embeddings.detach().cpu().numpy(),
        batch_size,
        np.random.default_rng(seed)
    )

    return selected_indices.tolist()
```

This is the pluggable component that implements **Algorithm 1, Steps 4-7** of the paper.

<a id="sec-5-running-active-learning-end-to-end"></a>

## 5. Running active learning end-to-end

We now run the full BADGE active learning loop. The protocol follows the paper's Section 4: start with 100 random labeled examples, then iteratively select batches of size 100 using BADGE, retrain from scratch, and evaluate on the test set.

<a id="sec-51-the-acquisition-loop"></a>

### 5.1 The acquisition loop

In [ ]:
def evaluate(model, x, y):
    """Evaluate test accuracy."""
    model.eval()
    with torch.no_grad():
        logits = model(x)
        return (logits.argmax(dim=1) == y).float().mean().item()

# Reset to initial state
labeled_idx = np.asarray(initial_indices)
unlabeled_idx = np.setdiff1d(np.arange(len(x_pool)), labeled_idx)

learning_curve = []

# Record initial accuracy
model = build_model(input_dim=input_dim, n_classes=n_classes, hidden_dim=cfg["hidden_dim"])
model = train_from_scratch(
    model, x_pool[labeled_idx], y_pool[labeled_idx],
    learning_rate=cfg["learning_rate"],
    max_epochs=cfg["max_epochs"],
    train_until_accuracy=cfg["train_until_accuracy"],
    seed=SEED
)
initial_acc = evaluate(model, x_test, y_test)
learning_curve.append((len(labeled_idx), initial_acc))
print(f"Round 0: {len(labeled_idx)} labels, test accuracy = {initial_acc:.4f}")

# Acquisition loop
for r in range(cfg["num_rounds"]):
    # Select batch from current unlabeled pool
    x_unlabeled = x_pool[unlabeled_idx]
    batch_positions = select_batch(model, x_unlabeled, cfg["batch_size"], SEED + r)
    
    # Map positions back to global indices
    chosen = unlabeled_idx[np.asarray(batch_positions)]
    
    # Update labeled/unlabeled sets (disjoint index sets)
    labeled_idx = np.union1d(labeled_idx, chosen)
    unlabeled_idx = np.setdiff1d(unlabeled_idx, chosen)
    
    # Retrain from scratch (FRESH model each round)
    model = build_model(input_dim=input_dim, n_classes=n_classes, hidden_dim=cfg["hidden_dim"])
    model = train_from_scratch(
        model, x_pool[labeled_idx], y_pool[labeled_idx],
        learning_rate=cfg["learning_rate"],
        max_epochs=cfg["max_epochs"],
        train_until_accuracy=cfg["train_until_accuracy"],
        seed=SEED + r
    )
    
    # Evaluate
    acc = evaluate(model, x_test, y_test)
    learning_curve.append((len(labeled_idx), acc))
    print(f"Round {r + 1}: {len(labeled_idx)} labels, test accuracy = {acc:.4f}")

<a id="sec-52-learning-curve"></a>

### 5.2 Learning curve

In [ ]:
labels_acquired = [lc[0] for lc in learning_curve]
accuracies = [lc[1] for lc in learning_curve]

plt.figure(figsize=(8, 5))
plt.plot(labels_acquired, accuracies, marker='o', linewidth=2, markersize=6)
plt.xlabel('Number of labels acquired', fontsize=12)
plt.ylabel('Test accuracy', fontsize=12)
plt.title(f'BADGE learning curve (MNIST, smoke scale)\n'
          f'batch_size={cfg["batch_size"]}, num_rounds={cfg["num_rounds"]}', fontsize=12)
plt.grid(True, alpha=0.3)
plt.xticks(labels_acquired)
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()

<a id="sec-6-use-your-own-data"></a>

## 6. Use your own data

To use your own dataset, you have two options:

**Option A: Pass an explicit path**
```python
x_pool, y_pool, x_test, y_test = load_data("path/to/your/data/")
```

**Option B: Drop your file in `method/example_data/` to override the default**
Place your file in the `method/example_data/` directory and call `load_data()` with no arguments — it will pick up your file automatically.

**Supported formats (priority order):**

1. **`.pt` file:** A single file containing a dict with keys `x_pool`, `y_pool`, `x_test`, `y_test`, each a torch.Tensor.
   ```python
   torch.save({
       "x_pool": x_pool_tensor,
       "y_pool": y_pool_tensor,
       "x_test": x_test_tensor,
       "y_test": y_test_tensor
   }, "my_data.pt")
   ```

2. **`.json` file:** Same structure as `.pt`, but values are nested lists (converted to tensors at load time).
   ```json
   {
     "x_pool": [[...], ...],
     "y_pool": [...],
     "x_test": [[...], ...],
     "y_test": [...]
   }
   ```

3. **`.csv` files:** Two files in the directory: `pool.csv` (or `train.csv`) and `test.csv`. Header row plus data rows; the **last column is the integer label**, all other columns are float features.

**Important:** Inputs (`x_pool`, `x_test`) must be 2D tensors of shape `(N, input_dim)`. The bundled MLP expects flattened features. For image-shaped data (e.g., `(N, C, H, W)`), either:
- Flatten the images before saving/loading, or
- Swap in a CNN architecture in `method/model.py` that handles 4D inputs and still provides the `forward_with_embedding` method.